# Compare models: stretch vs pad, ResNet18 vs EfficientNet-B0

EDA showed that normal and pneumonia images have different shapes (width/height 1.2 vs 1.5), so stretching every image to 224x224 squashes the two classes differently. Here I train 3 more models and compare them with the baseline (ResNet18 + stretch, already in `results/test_results.json`).

| Run | Model | Resize |
|---|---|---|
| baseline | ResNet18 | stretch |
| resnet18_pad | ResNet18 | pad |
| effnet_stretch | EfficientNet-B0 | stretch |
| effnet_pad | EfficientNet-B0 | pad |

Runtime → **T4 GPU**. Takes about 40 minutes.

In [ ]:
!git clone https://github.com/panidasrs/chest-xray-ai.git
%cd chest-xray-ai
!pip install -q kagglehub
!python prepare_data.py

## ResNet18 + pad

In [ ]:
!python train.py --arch resnet18 --resize pad --name resnet18_pad
!python evaluate.py --arch resnet18 --resize pad --name resnet18_pad

## EfficientNet-B0 + stretch

In [ ]:
!python train.py --arch efficientnet_b0 --resize stretch --name effnet_stretch
!python evaluate.py --arch efficientnet_b0 --resize stretch --name effnet_stretch

## EfficientNet-B0 + pad

In [ ]:
!python train.py --arch efficientnet_b0 --resize pad --name effnet_pad
!python evaluate.py --arch efficientnet_b0 --resize pad --name effnet_pad

## Comparison table (official test set, threshold chosen on validation)

In [ ]:
import json
import pandas as pd

runs = {
    "baseline": "results/test_results.json",
    "resnet18_pad": "results/resnet18_pad_test_results.json",
    "effnet_stretch": "results/effnet_stretch_test_results.json",
    "effnet_pad": "results/effnet_pad_test_results.json",
}
rows = []
for name, path in runs.items():
    r = json.load(open(path))
    r.setdefault("arch", "resnet18")    # the baseline file was made before these fields existed
    r.setdefault("resize", "stretch")
    rows.append({"run": name, **r})

table = pd.DataFrame(rows)[["run", "arch", "resize", "test_auc", "sensitivity", "specificity", "accuracy", "fn", "fp", "threshold"]]
table.to_csv("results/model_comparison.csv", index=False)
table

## Save results

Small result files go to GitHub. The model files (`models/*.pt`) are too big — copy them to Google Drive.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/chest-xray-models
!cp models/*.pt /content/drive/MyDrive/chest-xray-models/

!zip -j compare_results.zip results/model_comparison.csv results/*_test_results.json results/*_history.csv results/*_test_plots.png
files.download('compare_results.zip')